# MAI 600 Module 6 - Final Project Proposal Colab Starter

This notebook helps you create the Module 6 Final Project Proposal package. It includes:

- Project folder setup
- Safe sample documents
- Basic EDA / document review
- Optional simple RAG prototype with embeddings and FAISS
- Evaluation metrics plan
- Proposal files
- ZIP download for LMS or GitHub

Do not upload private, confidential, employer-sensitive, patient, legal, financial, proprietary, or protected information.

In [ ]:
!pip -q install pandas numpy matplotlib sentence-transformers faiss-cpu google-genai

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project_path = Path('/content/mai600-final-project-proposal')
folders = ['data', 'data/documents', 'docs', 'notebooks', 'results', 'images']
for folder in folders:
    (project_path / folder).mkdir(parents=True, exist_ok=True)
print(f'Project folder created at: {project_path}')

## 1. Define Your Project Idea

Edit these variables to match your own proposal.

In [ ]:
PROJECT_TITLE = 'IT Help Desk Policy Assistant with Citations'
FIELD = 'IT Help Desk / Business Operations'
PROBLEM = (
    'Support staff need quick answers from internal help desk policies, '
    'but policy information is spread across multiple documents.'
)
SELECTED_APPROACH = 'SLM + RAG'
EXPECTED_OUTPUT = 'Document-grounded answer with source citation and recommended next action'

print(PROJECT_TITLE)
print(SELECTED_APPROACH)

## 2. Create Safe Sample Documents

Use these fictional documents for the demo, or replace them with your own safe public/fictional documents.

In [ ]:
sample_docs = {
    'vpn_policy.md': '''# VPN Access Policy
Employees may request VPN access when they need to reach internal systems from outside the campus network. New VPN access requests require manager approval and identity verification. VPN accounts must use multi-factor authentication. Temporary VPN access expires after 14 days unless extended by the IT service desk. Support tickets for VPN issues should include the user name, device type, error message, and whether MFA was completed successfully.''',
    'password_reset_faq.md': '''# Password Reset FAQ
Users may reset their password using the self-service password portal. If the account is locked, the user should wait 15 minutes or contact the service desk. Service desk agents must verify the user's identity before manually resetting a password. Password reset tickets should not include full passwords or sensitive personal information. If MFA is unavailable, the ticket should be escalated to identity support.''',
    'support_sla.md': '''# IT Support SLA
Critical incidents affecting multiple users must be acknowledged within 15 minutes. High-priority tickets should be acknowledged within 1 hour. Standard service requests should be acknowledged within 1 business day. Tickets should include a clear problem summary, affected system, user impact, troubleshooting already attempted, and requested deadline. Escalation is required when the issue affects security, compliance, or a business-critical process.''',
    'data_handling_policy.md': '''# Support Data Handling Policy
Support teams must avoid storing sensitive data in plain text. Tickets should not include passwords, full Social Security numbers, payment card numbers, or private health information. When sensitive information appears in a ticket, agents should redact it and document only the minimum necessary details. AI tools may be used only with approved, non-sensitive, fictional, public, or properly anonymized data.''',
    'incident_escalation.md': '''# Incident Escalation Guide
An issue should be escalated when it affects multiple users, involves a security concern, creates a compliance risk, or blocks a critical business workflow. Escalated tickets should include severity, evidence, affected users, suspected system, timeline, and immediate workaround. The incident lead is responsible for communication updates and final resolution notes.''',
    'account_lockout.md': '''# Account Lockout Guide
A user account may be locked after repeated failed login attempts. Agents should confirm whether the issue affects one system or multiple systems. The recommended first step is to verify identity and check account status. If lockouts repeat after a reset, investigate saved credentials on mobile devices, browsers, and mapped drives.'''
}

for filename, text in sample_docs.items():
    (project_path / 'data' / 'documents' / filename).write_text(text, encoding='utf-8')
print('Sample documents created.')

## 3. Create a Data Inventory and EDA Summary

In [ ]:
records = []
for file_path in sorted((project_path / 'data' / 'documents').glob('*.md')):
    text = file_path.read_text(encoding='utf-8')
    words = text.split()
    records.append({
        'document_name': file_path.name,
        'document_type': 'Fictional policy / guide',
        'word_count': len(words),
        'estimated_tokens': round(len(words) * 1.33),
        'safe_to_use': 'Yes - fictional instructor-created data',
        'main_topic': file_path.stem.replace('_', ' ').title()
    })

sample_data = pd.DataFrame(records)
sample_data.to_csv(project_path / 'data' / 'sample_data.csv', index=False)
sample_data

In [ ]:
eda_summary = pd.DataFrame([
    {'Item': 'Number of documents', 'Result': len(sample_data)},
    {'Item': 'Document types', 'Result': ', '.join(sample_data['document_type'].unique())},
    {'Item': 'Average word count', 'Result': round(sample_data['word_count'].mean(), 1)},
    {'Item': 'Estimated total tokens', 'Result': int(sample_data['estimated_tokens'].sum())},
    {'Item': 'Sensitive data present?', 'Result': 'No - fictional classroom data'},
    {'Item': 'Main topics', 'Result': ', '.join(sample_data['main_topic'].tolist())}
])
eda_summary.to_csv(project_path / 'results' / 'initial_eda_summary.csv', index=False)
eda_summary

In [ ]:
data_description = f'''# Data Description

## Data Source
The project uses fictional instructor-created IT help desk policy documents.

## Data Type
The data consists of short markdown policy and procedure documents.

## What One Document Represents
Each document represents one policy, FAQ, or support guide that a help desk assistant may need to search.

## Number of Documents
{len(sample_data)} documents.

## Safety
The data is safe to use because it is fictional and does not contain private, confidential, employer-sensitive, patient, legal, financial, proprietary, or protected information.

## Important Fields or Sections
The files include policy names, requirements, escalation rules, support notes, and missing-information guidance.

## Known Limitations
The data collection is small and simplified for classroom demonstration. It does not represent a complete real-world IT knowledge base.

## Preprocessing Needed
The documents need to be loaded, cleaned, split into chunks, embedded, and indexed for retrieval.
'''

(project_path / 'data' / 'data_description.md').write_text(data_description, encoding='utf-8')
(project_path / 'docs' / 'data_description.md').write_text(data_description, encoding='utf-8')
print(data_description)

## 4. Optional Simple RAG Prototype

This is evidence for your proposal. It helps you explain why RAG may or may not fit your problem.

In [ ]:
def chunk_text(text, chunk_size=80, overlap=20):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(' '.join(words[start:end]))
        start += chunk_size - overlap
    return chunks

chunk_records = []
for file_path in sorted((project_path / 'data' / 'documents').glob('*.md')):
    text = file_path.read_text(encoding='utf-8')
    for i, chunk in enumerate(chunk_text(text, chunk_size=80, overlap=20)):
        chunk_records.append({
            'chunk_id': f'{file_path.stem}_chunk_{i+1}',
            'source_document': file_path.name,
            'chunk_number': i + 1,
            'chunk_text': chunk
        })
chunks_df = pd.DataFrame(chunk_records)
chunks_df.to_csv(project_path / 'results' / 'chunk_inventory.csv', index=False)
chunks_df.head()

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_texts = chunks_df['chunk_text'].tolist()
embeddings = embedding_model.encode(chunk_texts, normalize_embeddings=True)
embeddings = np.array(embeddings).astype('float32')
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f'Indexed {index.ntotal} chunks.')

In [ ]:
def retrieve(question, top_k=3):
    q_embedding = embedding_model.encode([question], normalize_embeddings=True)
    q_embedding = np.array(q_embedding).astype('float32')
    scores, indices = index.search(q_embedding, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = chunks_df.iloc[int(idx)].to_dict()
        row['similarity_score'] = float(score)
        results.append(row)
    return pd.DataFrame(results)

question = 'When should a support ticket be escalated?'
retrieved = retrieve(question, top_k=3)
retrieved[['source_document', 'chunk_id', 'similarity_score', 'chunk_text']]

In [ ]:
def build_rag_prompt(question, retrieved_df):
    context_blocks = []
    for i, row in retrieved_df.iterrows():
        citation = f"[{i+1}] {row['source_document']} :: {row['chunk_id']}"
        context_blocks.append(f"{citation}
{row['chunk_text']}")
    context_text = '

'.join(context_blocks)
    return f'''
You are a careful AI assistant. Answer the question using only the provided context.

Rules:
1. Use only the retrieved context.
2. If the answer is not supported, say "The provided documents do not contain enough information."
3. Include citations like [1], [2], or [3] for every factual claim.
4. Keep the answer concise and professional.

Retrieved context:
{context_text}

Question:
{question}

Answer:
'''

rag_prompt = build_rag_prompt(question, retrieved)
print(rag_prompt)
(project_path / 'results' / 'rag_prompt_for_manual_testing.txt').write_text(rag_prompt, encoding='utf-8')

### Optional Gemini Generation

Only run this if you have a Gemini API key. If not, copy the RAG prompt from above and paste it into an approved model manually.

In [ ]:
# Optional cell
import os
from getpass import getpass
from google import genai

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass('Enter your Gemini API key: ')

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
MODEL_NAME = 'gemini-3.5-flash'  # Replace with current Flash model if needed.

response = client.models.generate_content(model=MODEL_NAME, contents=rag_prompt)
print(response.text)

## 5. Evaluation Set and Retrieval Hit Rate

In [ ]:
eval_questions = pd.DataFrame([
    {'question_id': 'Q1', 'question': 'When should a support ticket be escalated?', 'expected_source': 'incident_escalation.md', 'expected_answer_summary': 'Escalate for multiple users, security concern, compliance risk, or critical workflow impact.'},
    {'question_id': 'Q2', 'question': 'What information should be included in a VPN support ticket?', 'expected_source': 'vpn_policy.md', 'expected_answer_summary': 'User name, device type, error message, and MFA status.'},
    {'question_id': 'Q3', 'question': 'What should agents avoid putting in support tickets?', 'expected_source': 'data_handling_policy.md', 'expected_answer_summary': 'Passwords, SSNs, payment card numbers, private health information, and unnecessary sensitive data.'},
    {'question_id': 'Q4', 'question': 'How long should a user wait after an account lockout?', 'expected_source': 'password_reset_faq.md', 'expected_answer_summary': 'Wait 15 minutes or contact the service desk.'},
    {'question_id': 'Q5', 'question': 'What is the acknowledgement target for critical incidents?', 'expected_source': 'support_sla.md', 'expected_answer_summary': 'Critical incidents affecting multiple users should be acknowledged within 15 minutes.'}
])
eval_questions.to_csv(project_path / 'results' / 'retrieval_eval_questions.csv', index=False)
eval_questions

In [ ]:
eval_results = []
TOP_K = 3
for _, row in eval_questions.iterrows():
    retrieved_df = retrieve(row['question'], top_k=TOP_K)
    retrieved_sources = retrieved_df['source_document'].tolist()
    hit = row['expected_source'] in retrieved_sources
    eval_results.append({
        'question_id': row['question_id'],
        'question': row['question'],
        'expected_source': row['expected_source'],
        'retrieved_sources': '; '.join(retrieved_sources),
        'retrieval_hit': int(hit),
        'top_k': TOP_K
    })

eval_results_df = pd.DataFrame(eval_results)
retrieval_hit_rate = eval_results_df['retrieval_hit'].mean()
eval_results_df.to_csv(project_path / 'results' / 'retrieval_eval_results.csv', index=False)
print(f'Retrieval hit rate: {retrieval_hit_rate:.2%}')
eval_results_df

In [ ]:
metrics_plan = pd.DataFrame([
    {'Metric': 'Retrieval hit rate', 'How measured': 'Correct source document appears in top-k retrieved chunks.', 'Target': 'At least 80% on initial test questions'},
    {'Metric': 'Citation accuracy', 'How measured': 'Human review checks whether each citation supports the sentence it is attached to.', 'Target': 'At least 4 out of 5 on a rubric'},
    {'Metric': 'Groundedness', 'How measured': 'Human reviewer checks whether the answer avoids unsupported claims.', 'Target': 'At least 4 out of 5 on a rubric'},
    {'Metric': 'Format adherence', 'How measured': 'Answer follows the required table or citation format.', 'Target': 'At least 90% of test answers follow format'}
])
metrics_plan.to_csv(project_path / 'results' / 'evaluation_metrics_plan.csv', index=False)
metrics_plan

## 6. Create Proposal and Submission Files

In [ ]:
proposal_text = f'''# Module 6 Final Project Proposal

## 1. Project Title
{PROJECT_TITLE}

## 2. Problem Definition
{PROBLEM}

The primary users are help desk staff, support managers, and employees who need fast answers from policy documents. If the problem is not solved, staff may spend extra time searching documents, provide inconsistent guidance, or miss escalation requirements.

## 3. Project Relevance
This project is relevant to {FIELD} because support teams depend on accurate, fast, and policy-grounded answers. A useful AI assistant could reduce search time, improve consistency, and help staff cite the source of each answer.

## 4. Dataset or Document Collection
The project uses a small collection of fictional IT help desk policy documents located in the data/documents folder.

Source: Instructor-created fictional documents.
Data type: Markdown policy, FAQ, and support guide documents.
Safety: The data is safe for classroom use because it is fictional and does not contain private, confidential, employer-sensitive, patient, legal, financial, proprietary, or protected information.

## 5. Background and Data Description
The data represents a simplified internal help desk knowledge base. Each document represents one policy, FAQ, or support procedure.

## 6. Proposed AI Approach
Selected approach: {SELECTED_APPROACH}

## 7. Approach Justification
This approach is stronger than simple prompting because the answer depends on information stored in policy documents. RAG is useful because it retrieves relevant source passages at question time and allows the answer to include citations. A local SLM may be useful if the organization wants privacy, cost control, or local experimentation. The trade-off is that local models may be slower or less capable than hosted models.

## 8. Expected System Output
The system will produce: {EXPECTED_OUTPUT}.

## 9. Baseline Model or Method
The baseline will be simple keyword search or prompting a local model without retrieval. The improved approach will retrieve relevant document chunks and use them as context before generating an answer.

## 10. Initial EDA or Data Review
See results/initial_eda_summary.csv and data/data_description.md.

## 11. Evaluation Metrics
The initial evaluation metrics are retrieval hit rate, citation accuracy, groundedness, and format adherence. See results/evaluation_metrics_plan.csv.

## 12. AI Tools Usage Plan
Planned tools include Google Colab, Sentence Transformers, FAISS, and optionally Gemini or a local SLM.

## 13. Risks, Limitations, and Ethics
Risks include hallucination, weak retrieval, missing documents, outdated policies, privacy exposure, and overreliance on model output.

## 14. References or Source Links
- Course Module 6 assignment materials
- Sentence Transformers documentation
- FAISS documentation
- Google Gemini API documentation, if used
'''
(project_path / 'proposal.md').write_text(proposal_text, encoding='utf-8')

readme_text = f'''# {PROJECT_TITLE}

## Problem
{PROBLEM}

## Proposed Approach
{SELECTED_APPROACH}

## Data
This project uses fictional instructor-created IT help desk policy documents. The data is safe for classroom use.

## Method
The proposed system loads documents, chunks text, creates embeddings, stores vectors in FAISS, retrieves relevant chunks, and uses an LLM to generate a citation-backed answer.

## Initial Evaluation Plan
Metrics include retrieval hit rate, citation accuracy, groundedness, and format adherence.

## AI Usage
See ai_usage_disclosure.md.
'''
(project_path / 'README.md').write_text(readme_text, encoding='utf-8')
print('proposal.md and README.md created.')

In [ ]:
(project_path / 'docs' / 'introduction.md').write_text(f'''# Introduction

{PROJECT_TITLE} is a proposed AI solution designed to help users answer questions from support policy documents with clear source references.
''', encoding='utf-8')

(project_path / 'docs' / 'background.md').write_text('''# Background

Support information is often spread across policies, FAQs, and troubleshooting guides. RAG can help by retrieving relevant source passages before answer generation.
''', encoding='utf-8')

ai_usage = '''# AI Usage Disclosure

## AI Tools Used
- ChatGPT:
- Claude:
- Gemini:
- Claude Code:
- GitHub Copilot:
- Hugging Face:
- Ollama:
- Other:

## How I Used AI
I used AI tools to brainstorm, review proposal language, troubleshoot notebook code, and compare possible AI approaches.

## Prompts Used
- Help me define an AI project problem statement.
- Help me improve evaluation metrics for a RAG proposal.

## What I Verified Myself
I reviewed data safety, checked retrieved chunks, and confirmed the final proposal choices.

## Limitations or Failures
AI suggestions may be incomplete or unsupported. I verified and edited the final work myself.

## Academic Integrity Statement
I confirm that AI was used as a learning and support tool, not as a replacement for my own work.
'''
(project_path / 'ai_usage_disclosure.md').write_text(ai_usage, encoding='utf-8')

(project_path / 'references.md').write_text('''# References

- Course Module 6 assignment materials.
- Sentence Transformers documentation.
- FAISS documentation.
- Google Gemini API documentation, if used.
''', encoding='utf-8')
print('Additional proposal files created.')

## 7. Create Architecture Image, Zip, and Download

In [ ]:
import matplotlib.pyplot as plt
steps = ['Documents', 'Chunking', 'Embeddings', 'Vector Search', 'Retrieved Context', 'LLM', 'Cited Answer']
plt.figure(figsize=(12, 3))
for i, label in enumerate(steps):
    plt.text(i, 0.5, label, ha='center', va='center', fontsize=10,
             bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='black'))
    if i < len(steps) - 1:
        plt.arrow(i + 0.22, 0.5, 0.5, 0, head_width=0.04, head_length=0.08, length_includes_head=True)
plt.axis('off')
plt.title('Proposed RAG Architecture')
plt.tight_layout()
plt.savefig(project_path / 'images' / 'proposed_architecture.png', dpi=150)
plt.show()

In [ ]:
import shutil
(project_path / 'notebooks' / 'README_notebook_note.md').write_text('Download your Colab notebook as initial_eda.ipynb and place it here before final submission.', encoding='utf-8')
zip_base = '/content/mai600-final-project-proposal-completed'
shutil.make_archive(zip_base, 'zip', project_path)
print('ZIP created:', zip_base + '.zip')

In [ ]:
from google.colab import files
files.download('/content/mai600-final-project-proposal-completed.zip')